In [1]:
import sys
sys.path.append("../..")  # raíz del repo, para poder importar src.*

import json
import os

import numpy as np
import pandas as pd
import tensorflow as tf

from src.models.gru.model import build_gru_model
from src.models.gru.train import compute_class_weights, train_gru_model
from src.models.gru.threshold import sweep_thresholds, select_threshold_strategies

np.random.seed(42)
tf.random.set_seed(42)

In [2]:
DATA_DIR = "../../data/sequences"
W = 10

X_train = np.load(f"{DATA_DIR}/X_train_w{W}.npy")
y_train = np.load(f"{DATA_DIR}/y_train_w{W}.npy")
X_val = np.load(f"{DATA_DIR}/X_val_w{W}.npy")
y_val = np.load(f"{DATA_DIR}/y_val_w{W}.npy")

print(f"X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape} | y_val:   {y_val.shape}")

X_train: (315222, 10, 63) | y_train: (315222,)
X_val:   (67513, 10, 63) | y_val:   (67513,)


In [3]:
class_weight_dict = compute_class_weights(y_train)
print("class_weight:", class_weight_dict)

class_weight: {0: 0.5131585373397712, 1: 19.499072126685636}


In [4]:
# baseline_32 se carga ya entrenado (no se reentrena, ver docs/gru/adr/0003-gru-early-stopping-criterion.md
# y docs/gru/adr/0005-ajuste-umbral-decision-gru.md para su origen); gru_64/gru_32_16 se entrenan aquí.
CONFIGURACIONES = {
    "baseline_32": {"tipo": "cargado", "path": "../../models/gru/baseline_w10.keras"},
    "gru_64": {"tipo": "nuevo", "units": 64},
    "gru_32_16": {"tipo": "nuevo", "units": 32, "second_layer_units": 16},
}

In [5]:
resultados = []

for nombre, config in CONFIGURACIONES.items():
    print(f"=== {nombre} ===")

    if config["tipo"] == "cargado":
        model = tf.keras.models.load_model(config["path"])
        epocas_entrenadas = None
    else:
        model = build_gru_model(
            input_shape=(W, X_train.shape[-1]),
            units=config.get("units", 32),
            second_layer_units=config.get("second_layer_units"),
        )
        history = train_gru_model(
            model, X_train, y_train, X_val, y_val, class_weight_dict,
            epochs=50, batch_size=64, random_state=42,
        )
        epocas_entrenadas = len(history.history["loss"])

        model_path = f"../../models/gru/hp_w10_{nombre}.keras"
        os.makedirs(os.path.dirname(model_path), exist_ok=True)
        model.save(model_path)

    val_loss = model.evaluate(X_val, y_val, verbose=0)[0]

    df_umbrales = sweep_thresholds(model, X_val, y_val)
    estrategias = select_threshold_strategies(df_umbrales)

    resultados.append({
        "configuracion": nombre,
        "val_loss": val_loss,
        "f2_maximo": estrategias["f2_maximo"]["f2"],
        "umbral_f2_maximo": estrategias["f2_maximo"]["threshold"],
        "f1_maximo": estrategias["f1_maximo"]["f1"],
        "umbral_f1_maximo": estrategias["f1_maximo"]["threshold"],
        "epocas_entrenadas": epocas_entrenadas,
        "parametros_totales": model.count_params(),
    })

=== baseline_32 ===
=== gru_64 ===
Epoch 1/50
4926/4926 ━━━━━━━━━━━━━━━━━━━━ 14s 3ms/step - accuracy: 0.9956 - auc: 0.9993 - loss: 0.0188 - precision: 0.8582 - recall: 0.9936 - val_accuracy: 0.9948 - val_auc: 0.9998 - val_loss: 0.0132 - val_precision: 0.8594 - val_recall: 0.9972
Epoch 2/50
4926/4926 ━━━━━━━━━━━━━━━━━━━━ 13s 3ms/step - accuracy: 0.9990 - auc: 0.9998 - loss: 0.0056 - precision: 0.9637 - recall: 0.9985 - val_accuracy: 0.9990 - val_auc: 1.0000 - val_loss: 0.0032 - val_precision: 0.9794 - val_recall: 0.9896
Epoch 3/50
4926/4926 ━━━━━━━━━━━━━━━━━━━━ 13s 3ms/step - accuracy: 0.9992 - auc: 0.9999 - loss: 0.0032 - precision: 0.9727 - recall: 0.9988 - val_accuracy: 0.9968 - val_auc: 0.9998 - val_loss: 0.0056 - val_precision: 0.9670 - val_recall: 0.9298
Epoch 4/50
4926/4926 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step - accuracy: 0.9994 - auc: 1.0000 - loss: 0.0024 - precision: 0.9777 - recall: 0.9989 - val_accuracy: 0.9984 - val_auc: 0.9999 - val_loss: 0.0039 - val_precision: 0.9762 - val_

In [6]:
tabla_comparativa = pd.DataFrame(resultados)
print(tabla_comparativa)

  configuracion  val_loss  f2_maximo  umbral_f2_maximo  f1_maximo  \
0   baseline_32  0.001350   0.996690             0.050   0.994554   
1        gru_64  0.003203   0.992161             0.438   0.986347   
2     gru_32_16  0.008940   0.966499             0.323   0.972182   

   umbral_f1_maximo  epocas_entrenadas  parametros_totales  
0             0.337                NaN                9345  
1             0.478                7.0               24833  
2             0.454                9.0               11729  


In [7]:
RESULTS_PATH = "../../results/gru/hyperparameter_search_w10.json"
os.makedirs(os.path.dirname(RESULTS_PATH), exist_ok=True)

with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(resultados, f, indent=2, ensure_ascii=False)

print(f"Resultados guardados en: {RESULTS_PATH}")

Resultados guardados en: ../../results/gru/hyperparameter_search_w10.json


## Regla de decisión

Una variante reemplaza a `models/gru/baseline_w10.keras` como baseline oficial solo si,
en validación, mejora el F2 máximo en al menos 0.001 **Y** reduce val_loss frente a
`baseline_32`. Si ninguna configuración cumple ambas condiciones a la vez, se conserva
el baseline actual sin cambios.